# Predição: Prova de Fotos - Comparação MSOV1 e MSOV2

Autor:  Viviane da Rosa Sommer

Atualizado: 09/01/2025

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (MSO-V1 e MSO-V2) nas fotos dos vídeos das provas de fogo.

Foi feita uma amostragem de 30 imagens positivas e 30 imagens negativas para cada uma das provas de foto.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import glob
import random
import cv2
import numpy as np
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torchvision
import torch
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800
SAMPLE_SIZE = 30

mso_v1 = YOLO('../Coral_Note/runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')
mso_v2 = YOLO('runs/segment/train3/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Reads and crops an image from a file.

    Args:
        filename (str): Path to the image file.

    Returns:
        np.ndarray: Cropped image as a NumPy array.
    """
    image = cv2.imread(filename)
    if image is None:
        raise ValueError(f"Unable to load image: {filename}")
    height, _, _ = image.shape
    mid = height // 2
    crop_range = int(0.34 * height)
    top, bottom = max(0, mid - crop_range), min(height, mid + crop_range)
    return image[top:bottom, :]


def process_results(results: list) -> any:
    """
    Extracts the first valid result containing masks from the model's output.

    Args:
        results (list): List of model detection results.

    Returns:
        any: The first valid result with masks, or None if no valid result exists.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.array, model) -> np.ndarray:
    """
    Predicts coral regions in an image using a given model.

    Args:
        img (np.array): Input image.
        model: Model to use for prediction.

    Returns:
        np.ndarray: Binary mask for coral regions.
    """
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    if coral_best_result and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_masks) > 0:
            nms_indices = torchvision.ops.nms(boxes[coral_indices, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int().cpu().numpy() * 255
        else:
            final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    else:
        final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)

    return cv2.resize(final_coral_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)


def generate_predictions(files: list, model_v1, model_v2) -> list:
    """
    Generates predictions for all files using two models.

    Args:
        files (list): List of image file paths.
        model_v1: First model.
        model_v2: Second model.

    Returns:
        list: List of dictionaries containing predictions.
    """
    results = []
    for filename in files:
        try:
            img = read_image(filename)
            mask_v1 = prediction_coral(img, model_v1)
            mask_v2 = prediction_coral(img, model_v2)
            results.append({"file": filename, "image": img, "mask_v1": mask_v1, "mask_v2": mask_v2})
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    return results


def plot_image_comparisons(results: list) -> None:
    """
    Plots image comparisons of predictions for all files.

    Args:
        results (list): List of dictionaries containing predictions.

    Returns:
        None: Displays the plots.
    """
    for result in results:
        try:
            img = result["image"]
            mask_v1 = result["mask_v1"]
            mask_v2 = result["mask_v2"]

            folder_name = os.path.basename(os.path.dirname(result["file"]))
            label = "Positive" if "POSITIVA" in folder_name.upper() else "Negative"
            
            annotated_img_v1 = img.copy()
            annotated_img_v2 = img.copy()

            contours_v1, _ = cv2.findContours(mask_v1.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contours_v2, _ = cv2.findContours(mask_v2.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            cv2.drawContours(annotated_img_v1, contours_v1, -1, (0, 0, 255), thickness=4)  # Red for mso_v1
            cv2.drawContours(annotated_img_v2, contours_v2, -1, (0, 255, 0), thickness=4)  # Green for mso_v2

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            annotated_v1_rgb = cv2.cvtColor(annotated_img_v1, cv2.COLOR_BGR2RGB)
            annotated_v2_rgb = cv2.cvtColor(annotated_img_v2, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 10))

            plt.subplot(1, 3, 1)
            plt.imshow(img_rgb)
            plt.title(f"Original Image ({label})")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(annotated_v1_rgb)
            plt.title("Model mso_v1 Prediction")
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(annotated_v2_rgb)
            plt.title("Model mso_v2 Prediction")
            plt.axis('off')

            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Error plotting for {result['file']}: {e}")
            continue


def evaluate_predictions(results: list, labels: list) -> None:
    """
    Evaluates predictions using confusion matrices and classification reports.

    Args:
        results (list): List of dictionaries containing predictions.
        labels (list): Ground truth labels for each image.

    Returns:
        None: Displays evaluation results.
    """
    predictions_v1 = [1 if result["mask_v1"].any() else 0 for result in results]
    predictions_v2 = [1 if result["mask_v2"].any() else 0 for result in results]

    conf_matrix_v1 = confusion_matrix(labels, predictions_v1)
    conf_matrix_v2 = confusion_matrix(labels, predictions_v2)

    print("\nClassification Report for mso_v1:")
    print(classification_report(labels, predictions_v1, target_names=["Negative", "Positive"]))

    print("\nClassification Report for mso_v2:")
    print(classification_report(labels, predictions_v2, target_names=["Negative", "Positive"]))

    plot_confusion_matrices_side_by_side(
        conf_matrices=[conf_matrix_v1, conf_matrix_v2],
        model_names=["mso_v1", "mso_v2"],
        class_names=["Negative", "Positive"]
    )


def plot_confusion_matrices_side_by_side(conf_matrices: list, model_names: list, class_names: list) -> None:
    """
    Plots multiple confusion matrices side by side.

    Args:
        conf_matrices (list): List of confusion matrices (numpy arrays).
        model_names (list): List of model names corresponding to each confusion matrix.
        class_names (list): List of class names (e.g., ["Negative", "Positive"]).

    Returns:
        None: Displays the confusion matrices in a single figure.
    """
    num_matrices = len(conf_matrices)
    plt.figure(figsize=(6 * num_matrices, 6))

    for i, (conf_matrix, model_name) in enumerate(zip(conf_matrices, model_names)):
        plt.subplot(1, num_matrices, i + 1)
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")

    plt.tight_layout()
    plt.show()

## Prova de Fotos - OS 6000696049

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto _6000696049/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto _6000696049/NEGATIVA"

positive_image_files = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.png")
negative_image_files = glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/*.png")

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

all_files = positive_sampled_files + negative_sampled_files
all_labels = [1] * len(positive_sampled_files) + [0] * len(negative_sampled_files)

results = generate_predictions(all_files, mso_v1, mso_v2)

plot_image_comparisons(results)

evaluate_predictions(results, all_labels)

## Prova de Fotos - OS 6000504195

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto_6000504195/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de foto_6000504195/NEGATIVA"

positive_image_files = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.png")
negative_image_files = glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/*.png")

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

all_files = positive_sampled_files + negative_sampled_files
all_labels = [1] * len(positive_sampled_files) + [0] * len(negative_sampled_files)

results = generate_predictions(all_files, mso_v1, mso_v2)

plot_image_comparisons(results)

evaluate_predictions(results, all_labels)

## Prova de Fotos - OS 6000701944

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de Foto_6000701944/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Provas de Fogo/Provas-de-foto/Prova de Foto_6000701944/NEGATIVA"

positive_image_files = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.png")
negative_image_files = glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/*.png")

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

all_files = positive_sampled_files + negative_sampled_files
all_labels = [1] * len(positive_sampled_files) + [0] * len(negative_sampled_files)

results = generate_predictions(all_files, mso_v1, mso_v2)

plot_image_comparisons(results)

evaluate_predictions(results, all_labels)

In [ ]:
!jupyter nbconvert --to html Prova-Foto-MSOV1-MSOV2.ipynb